# 12 Causal Inference and Policy Evaluation

Does shower exposure really "cause" infection? Does disinfecting the water system really work?

Workflow: **DAG causal diagram → confounder/mediator/collider → attributable risk AR/PAR → DiD intervention evaluation → parallel trends check**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — DAG: Untangle the Causal Structure with Text/Diagrams, No graphviz Needed

The first step in causal inference isn't running a regression — it's **drawing out the causal relationships you believe exist**. Here we hand-draw nodes and arrows with matplotlib, so there's no need to install graphviz separately (it often fails to install on Colab or CI, and every extra system dependency is another place things can break). Once the diagram is drawn, the key is to recognize three roles: a **confounder** (affects both the exposure and the outcome), a **mediator** (the exposure affects the outcome only through it), and a **collider** (pointed to by two variables at once).

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `nodes = {...}` | Defines the box position of every variable in the DAG using a coordinate dictionary |
> | `for src, dst in arrows: ax.annotate(...)` | Draws arrows in the causal direction; whatever an arrow points to is the "effect" |
> | `print("Confounder: functional_status → shower_use and → infection")` | Identifies the confounder: a variable that points to both the exposure and the outcome |
> | `print("Collider: hospitalized ← severity and ← infection")` | Identifies the collider: a variable pointed into by two arrows at once; adjusting for it actually creates a spurious association |

> 🧭 **Three roles, two opposite treatments**: confounders should be adjusted for (that's exactly what Ch05's stratified analysis and Ch06's regression adjustment do); mediators usually should not be adjusted for (adjusting for one "switches off" the causal pathway itself and underestimates the total effect); colliders must never be adjusted for — conditioning on a collider opens up a path that didn't exist before, producing a spurious association (collider bias).


In [ ]:
# --- Step 1: DAG (directed acyclic graph) ---
# Describe causal relationships with text (no need to install graphviz)
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# DAG visualization
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

# Nodes
nodes = {
    "floor/wing": (1, 5.5),
    "water\ncontamination": (3.5, 5.5),
    "shower\naerosol": (6, 5.5),
    "infection": (8.5, 5.5),
    "functional\nstatus": (1, 3),
    "shower\nuse": (4, 3),
    "age": (1, 1),
    "comorbidities": (3.5, 1),
    "severity": (6, 1),
    "death": (8.5, 1),
}

for name, (x, y) in nodes.items():
    ax.add_patch(plt.Rectangle((x-0.7, y-0.4), 1.4, 0.8,
                 fill=True, facecolor="#e0e0e0", edgecolor="black", linewidth=1.5))
    ax.text(x, y, name, ha="center", va="center", fontsize=8, fontweight="bold")

# Arrows (causal direction)
arrows = [
    ("floor/wing", "water\ncontamination"),
    ("water\ncontamination", "shower\naerosol"),
    ("shower\naerosol", "infection"),
    ("functional\nstatus", "shower\nuse"),
    ("shower\nuse", "infection"),
    ("functional\nstatus", "infection"),
    ("age", "comorbidities"),
    ("comorbidities", "severity"),
    ("severity", "death"),
    ("infection", "severity"),
]

for src, dst in arrows:
    x1, y1 = nodes[src]
    x2, y2 = nodes[dst]
    ax.annotate("", xy=(x2-0.7, y2), xytext=(x1+0.7, y1),
                arrowprops=dict(arrowstyle="->", color="#333", lw=1.5))

ax.set_title("Legionella DAG — Causal Diagram", fontsize=14)
plt.tight_layout()
plt.show()

print("=== Identifying Causal Structures ===")
print("Confounder: functional_status → shower_use and → infection")
print("Mediator: shower_aerosol lies between water_contamination → infection")
print("Collider: hospitalized ← severity and ← infection")
print("\n→ Controlling for the confounder (done in Ch05) = correct")
print("→ Controlling for the collider = wrong! It creates a spurious association")

## Step 2 — Attributable Risk (AR): How Much Extra Risk Does the Exposure Carry?

Attributable Risk (AR) answers how much the absolute risk differs between the "exposed" group and the "unexposed" group — note that it's a **subtraction**, not a division (division gives the risk ratio RR, which we already computed in Ch03). Here we reuse the `shower_use` exposure grouping from Ch05/Ch06, recompute the attack rate, and subtract.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `exposed = df[df["shower_use"] == 1]` | Filters to the exposed group ("showered") |
> | `risk_exposed = exposed["infected"].mean()` | The exposed group's attack rate (mean of a 0/1 column = proportion) |
> | `AR = risk_exposed - risk_unexposed` | Attributable risk: the absolute extra risk the exposed group carries over the unexposed group |
> | `PAR = risk_total - risk_unexposed` | Population attributable risk: swap the denominator for the whole population, to estimate how much risk this exposure contributes to the entire population |

> 💡 **AR and RR are two ways of asking about the same data**: RR answers "how many times higher is the risk", AR answers "how many percentage points higher". The two carry different meanings for public health decisions — a large RR with a small AR means that, although the relative risk is high, the actual number of preventable cases is limited (common for rare diseases).


In [ ]:
# --- Step 2: Attributable Risk ---
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Attack rate for shower exposure
exposed = df[df["shower_use"] == 1]
unexposed = df[df["shower_use"] == 0]

risk_exposed = exposed["infected"].mean()
risk_unexposed = unexposed["infected"].mean()
risk_total = df["infected"].mean()

print("=== Attack Rate for Shower Exposure ===")
print(f"Attack rate among showerers: {risk_exposed:.1%} ({exposed['infected'].sum()}/{len(exposed)})")
print(f"Attack rate among non-showerers: {risk_unexposed:.1%} ({unexposed['infected'].sum()}/{len(unexposed)})")
print(f"Overall attack rate: {risk_total:.1%}")

# Attributable Risk
AR = risk_exposed - risk_unexposed
print(f"\n=== Attributable Risk (AR) ===")
print(f"AR = {risk_exposed:.3f} - {risk_unexposed:.3f} = {AR:.3f}")
print(f"→ Showerers have {AR:.1%} more infection risk than non-showerers")

# Population Attributable Risk
PAR = risk_total - risk_unexposed
PAR_pct = PAR / risk_total * 100
print(f"\n=== Population Attributable Risk (PAR) ===")
print(f"PAR = {risk_total:.3f} - {risk_unexposed:.3f} = {PAR:.3f}")
print(f"PAR% = {PAR_pct:.1f}%")
print(f"→ If shower exposure were eliminated, infections could theoretically drop by {PAR_pct:.0f}%")
print("→ Assumptions: the causal relationship holds, and there are no other transmission routes")

## Step 2b — Two Ways to Compute PAF: Levin's Formula vs. Direct Subtraction, and They Must Agree

The PAR in the previous step was the absolute version, computed by subtracting "risks". Here we switch to **Levin's formula**, which needs only two numbers — the exposure prevalence `Pe` and the risk ratio `RR` — to compute the **Population Attributable Fraction** (PAF), which is the percentage version of PAR divided by the overall risk. The two methods are theoretically equivalent, and here we compute both to cross-check each other.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `Pe = (df["shower_use"] == 1).mean()` | Exposure prevalence: the proportion of all residents who showered |
> | `RR = risk_exposed / risk_unexposed` | Risk ratio, reusing the two risks computed in Step 2 |
> | `PAF_levin = Pe * (RR - 1) / (1 + Pe * (RR - 1))` | Levin's formula: derives the population attributable fraction from just Pe and RR |
> | `PAF_alt = (risk_total - risk_unexposed) / risk_total` | An equivalent alternative: the relative difference between "overall risk" and "unexposed risk" directly |

> 🧭 PAF = "the proportion of cases that could theoretically be eliminated if the whole population were unexposed" — **this interpretation only holds if the relationship is causal**; if `shower_use` is merely a bystander that happens to co-occur with the real cause (e.g., water system contamination), eliminating showers would not actually make PAF% of the cases disappear.


In [ ]:
# --- Step 2b: Population Attributable Fraction (Levin formula vs. the I_total form) ---
# Population Attributable Fraction PAF: Levin formula vs (I_total - I_unexposed)/I_total -- the two are equivalent
Pe = (df["shower_use"] == 1).mean()          # exposure prevalence
RR = risk_exposed / risk_unexposed
PAF_levin = Pe * (RR - 1) / (1 + Pe * (RR - 1))
risk_total = df["infected"].mean()
PAF_alt = (risk_total - risk_unexposed) / risk_total

print("=== Population Attributable Fraction PAF (cross-checked two ways) ===")
print(f"Exposure prevalence Pe = {Pe:.1%}, risk ratio RR = {RR:.2f}")
print(f"PAF (Levin)      = {PAF_levin:.1%}")
print(f"PAF (I_tot form) = {PAF_alt:.1%}   agree = {abs(PAF_levin - PAF_alt) < 1e-9}")

## Step 2c — A Cleaner Example: A Food Poisoning 2×2 Table (Illustrative Numbers)

The legionella dataset has only 280 rows and a fairly weak signal, so the AR/PAF numbers aren't especially "clean". Here we switch to a set of **illustrative teaching numbers** (not a real study, not a real event) and practice the same formulas on the most classic 2×2 table format, to confirm you can actually do the calculation rather than just copy-paste it.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `a, b, c, d = 120, 280, 25, 375` | Illustrative teaching numbers: a=exposed+ill, b=exposed+not ill, c=unexposed+ill, d=unexposed+not ill |
> | `risk_exposed_fp = a / (a + b)` | Risk of illness in the exposed group |
> | `risk_unexposed_fp = c / (c + d)` | Risk of illness in the unexposed group |
> | `Pe_fp = (a + b) / (a + b + c + d)` | Exposure prevalence |
> | `PAF_fp = Pe_fp * (RR_fp - 1) / (1 + Pe_fp * (RR_fp - 1))` | Apply the same Levin formula |

> ⚠️ These are **illustrative teaching numbers**, not statistics from any real food poisoning event — this particular set of numbers was chosen purely because it works out to AR≈23.7%, RR≈4.80, PAF≈65.5%, which are conveniently memorable magnitudes for checking your work. For real-world outbreak data, always go back to `legionella_outbreak.csv`.


In [ ]:
# --- Step 2c: Illustrative 2x2 table -- food poisoning AR/RR/PAF (not a real study, teaching numbers) ---
# 2x2 table: a=exposed+ill, b=exposed+not ill, c=unexposed+ill, d=unexposed+not ill
a, b, c, d = 120, 280, 25, 375  # illustrative teaching numbers, not a real event

risk_exposed_fp = a / (a + b)        # risk of illness in the exposed group
risk_unexposed_fp = c / (c + d)      # risk of illness in the unexposed group
AR_fp = risk_exposed_fp - risk_unexposed_fp
RR_fp = risk_exposed_fp / risk_unexposed_fp
Pe_fp = (a + b) / (a + b + c + d)    # exposure prevalence
PAF_fp = Pe_fp * (RR_fp - 1) / (1 + Pe_fp * (RR_fp - 1))

print("=== Illustrative example: food poisoning 2x2 table (not a real study) ===")
print(f"Exposed group risk = {a}/{a + b} = {risk_exposed_fp:.3f}")
print(f"Unexposed group risk = {c}/{c + d} = {risk_unexposed_fp:.3f}")
print(f"AR  = {AR_fp:.3f}  ({AR_fp:.1%})")
print(f"RR  = {RR_fp:.2f}")
print(f"PAF = {PAF_fp:.1%}")

## Step 3 — Counterfactual Thinking: What Would Happen "If There Had Been No Exposure"?

PAR already implies counterfactual reasoning; here we spell it out more explicitly: suppose **every resident had the same risk as the unexposed group** — how many people would theoretically be infected? The gap between that number and the actual number infected is the "theoretically preventable" case count.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `counterfactual_cases = int(n_total * risk_unexposed)` | Counterfactual scenario: expected number infected if everyone had the unexposed group's risk |
> | `prevented = n_infected - counterfactual_cases` | The gap between the actual and counterfactual expected infection counts = the theoretically preventable case count |

> ⚠️ Counterfactual estimates are always "what if", never "what actually happened" — the assumption still requires a causal relationship between exposure and outcome, and completely eliminating the exposure (e.g., banning all residents from showering) is often infeasible and inhumane in practice. A more realistic approach is the kind of DiD analysis starting in Step 4: improve the "safety" of the exposure (e.g., disinfecting the water system) instead of banning the exposure itself.


In [ ]:
# --- Step 3: Counterfactual Thinking ---
n_total = len(df)
n_infected = df["infected"].sum()

# Counterfactual: if no one had showered
counterfactual_cases = int(n_total * risk_unexposed)
prevented = n_infected - counterfactual_cases

print("=== Counterfactual Scenario ===")
print(f"Actual number infected: {n_infected}")
print(f"If no one had showered (counterfactual): {counterfactual_cases} expected infections")
print(f"Preventable infections: {prevented}")
print(f"\n→ But this is only a theoretical estimate!")
print("→ In practice, banning everyone from showering is not feasible")
print("→ A more realistic approach: disinfect the water system so showering becomes safe")

## Step 4 — DiD Data Preparation: Reshape Cases into a "Two Groups × Daily" Long Panel

The data shape DiD (Difference-in-Differences) needs is different from the previous steps: instead of one row per person, it's a long-format **panel** of "treated/control group" × "each day". Here we first aggregate cases by date, fill missing dates with 0 (no cases doesn't mean no data — you can't just drop those rows), then assemble the `treated` and `post` binary columns that a DiD regression can't do without.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]` | Defines the treated group: floors 2-3, Wing B (the target area for the water-system disinfection) |
> | `.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)` | Aggregates case counts by date, filling missing dates with 0 so every day has a row |
> | `"treated": [1] * len(all_dates) + [0] * len(all_dates)` | Group flag: 1 = treated group, 0 = control group |
> | `panel["post"] = (panel["date"] >= "2026-01-25").astype(int)` | Time flag: 1 = after the intervention, 0 = before; together with `treated` these are the two pillars of the DiD regression |
> | `panel["day"] = (panel["date"] - panel["date"].min()).dt.days` | Converts dates to a numeric day count, handy for plotting or trend analysis |

> 🧭 This panel is the core data structure of DiD: each row is the case count for "one group, one day", and the two binary columns `treated` and `post` cross into exactly four combinations (treated/control × before/after). Step 6's regression is essentially comparing the averages of these four cells.


In [ ]:
# --- Step 4: DiD Data Preparation ---
# Scenario: on January 25, emergency water-system disinfection was carried out for Wing B on floors 2-3
# Treated group: Wing B on floors 2-3 (high attack rate, close to the contamination source)
# Control group: all of floor 1 + Wing A on floors 2-3 (different water supply or non-target area)

df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
cases = df[df["infected"] == 1].copy()

# Build daily panel data
all_dates = pd.date_range("2026-01-12", "2026-01-28", freq="D")

# Treated group: Wing B on floors 2-3
treated_cases = cases[(cases["floor"].isin([2, 3])) & (cases["wing"] == "B")]
# reindex fills 0: dates with no cases can't just vanish -- DiD needs every day to have a row
treated_daily = treated_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Control group: the remaining areas
control_cases = cases[~((cases["floor"].isin([2, 3])) & (cases["wing"] == "B"))]
control_daily = control_cases.groupby("symptom_onset_date").size().reindex(all_dates, fill_value=0)

# Assemble into a long panel: one row for each (group, day)
panel = pd.DataFrame({
    "date": list(all_dates) * 2,
    "treated": [1] * len(all_dates) + [0] * len(all_dates),  # 1=treated, 0=control
    "daily_cases": list(treated_daily.values) + list(control_daily.values),
})
# post and treated are the two binary variables the DiD regression (Step 6) can't do without
panel["post"] = (panel["date"] >= "2026-01-25").astype(int)  # 1=after intervention, 0=before
panel["day"] = (panel["date"] - panel["date"].min()).dt.days  # numeric day count, handy for plotting/trend analysis

print("=== DiD Panel Data ===")
print(f"Treated group (Wing B, floors 2-3): {len(treated_daily)} days")
print(f"Control group (remaining areas): {len(control_daily)} days")
print(f"Intervention date: 2026-01-25")
print(f"\nCase counts before vs after intervention:")
summary = panel.groupby(["treated", "post"]).agg(total=('daily_cases','sum'), mean=('daily_cases','mean')).reset_index()
summary["group"] = summary["treated"].map({1: "Treated (Wing B, 2-3F)", 0: "Control"})
summary["period"] = summary["post"].map({0: "Before", 1: "After"})
print(summary[["group", "period", "total", "mean"]].to_string(index=False))

## Step 5 — Parallel Trends Check: Look Before You Trust a DiD Estimate

A DiD estimate is only credible if the **parallel trends assumption** holds: if the two groups were already on different slopes before the intervention, the post-intervention gap might just be the groups continuing to diverge on their own, unrelated to the intervention itself. Plotting is the most intuitive way to check this assumption — look first, trust the model second.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `ax.plot(all_dates, treated_daily.values, ...)` | Plots the treated group's daily case count trend line |
> | `ax.plot(all_dates, control_daily.values, ...)` | Plots the control group's trend line on the same chart for comparison |
> | `ax.axvline(x=pd.Timestamp("2026-01-25"), ...)` | Marks the intervention date, splitting the time axis into "before (left side, used to check parallel trends)" and "after (right side, used to observe the effect)" |

> ⚠️ Look only at the **left side** of the intervention line: if the two lines rise and fall roughly in parallel (even if the levels differ, that's fine), the parallel trends assumption holds up; if the two lines were already diverging — one rising, one falling — before the intervention, the DiD estimate can't simply be read as "the effect of the intervention".


In [ ]:
# --- Step 5: Parallel Trends Check + DiD Visualization ---
fig, ax = plt.subplots(figsize=(10, 5))

# Parallel trends check:
# Look only at the left side of the intervention date (dashed line) -- do the treated and control lines
# rise/fall roughly in parallel?
# If they are not parallel before the intervention, the two groups already had different rates of
# change, and the DiD estimate will be distorted

# Treated group
ax.plot(all_dates, treated_daily.values, marker="o", markersize=4,
        label="Treated (Wing B, 2-3F)", color="#e34a33")
# Control group
ax.plot(all_dates, control_daily.values, marker="s", markersize=4,
        label="Control (rest)", color="#2c7fb8")

# Intervention line: splits the time axis into "before (left side, used to check parallel trends)"
# and "after (right side, used to observe the effect)"
ax.axvline(x=pd.Timestamp("2026-01-25"), color="black", linestyle="--",
           alpha=0.7, label="Intervention date (1/25)")

ax.set_title("DiD — Case Count Trends Before and After Intervention")
ax.set_xlabel("Date")
ax.set_ylabel("Daily case count")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("→ Check whether the two lines are roughly parallel before the intervention (left of the dashed line)")
print("→ If they are parallel, the DiD estimate is more trustworthy")

## Step 6 — DiD OLS: The `treated:post` Interaction Term Is the Answer

A DiD regression looks like just one line of `smf.ols(...)`, but everything that matters is hidden in the **interaction term** `treated:post`. `treated` and `post` on their own only control for fixed differences in "group" and "time"; only `treated:post` captures how much more (or less) the treated group changed after the intervention, beyond what "group differences" and "time trends" alone can explain — and that's what DiD is actually estimating as the intervention effect.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `smf.ols("daily_cases ~ treated + post + treated:post", data=panel)` | `treated` = group main effect, `post` = time main effect, `treated:post` = interaction term (the DiD estimate) |
> | `.fit(cov_type="HC3")` | Uses HC3-robust standard errors to handle the heteroskedasticity common in panel data, avoiding overly optimistic significance tests |
> | `did_effect = model.params["treated:post"]` | Extracts the interaction coefficient — the "extra effect the intervention had on the treated group" |
> | `did_p = model.pvalues["treated:post"]` | Whether this effect is statistically significant also comes from the `treated:post` term, not `treated` or `post` |

> 💡 When reading DiD regression output, keep your eyes locked on the `treated:post` row — the coefficients for `treated` and `post` usually aren't what we care about; they're just there so the model can "net out" the fixed differences in group and time, letting the interaction term cleanly represent the intervention effect.


In [ ]:
# --- Step 6: DiD OLS Regression ---
# treated      -> group main effect (treated vs control, regardless of time)
# post         -> time main effect (before vs after, regardless of group)
# treated:post -> interaction term = the DiD estimate: how much extra (or less) change the
#                 treated group had after the intervention
# cov_type="HC3": panel data commonly has heteroskedasticity; robust SEs avoid overly optimistic p-values
model = smf.ols("daily_cases ~ treated + post + treated:post", data=panel).fit(cov_type="HC3")

print("=== DiD Regression Results (HC3-robust SEs) ===")
print(model.summary().tables[1])

# Only look at the treated:post term -- this is the actual DiD estimate of the intervention effect
did_effect = model.params["treated:post"]
did_p = model.pvalues["treated:post"]

print(f"\n=== DiD Effect Estimate ===")
print(f"treated:post coefficient = {did_effect:.3f}")
print(f"p-value = {did_p:.4f}")

if did_effect < 0:
    print(f"\n→ After the intervention, the treated group had {abs(did_effect):.1f} fewer daily cases than expected")
else:
    print(f"\n→ After the intervention, the treated group had {did_effect:.1f} more daily cases than expected")

if did_p < 0.05:
    print("→ The effect is statistically significant (p < 0.05)")
else:
    print("→ The effect is not statistically significant (p ≥ 0.05)")
    print("→ Possible reasons: insufficient sample size, too short an observation window, or the effect needs more time to appear")

## Summary

| Step | Skill Learned |
|------|------------|
| DAG | Use diagrams to identify confounders, mediators, and colliders |
| AR | The absolute extra infection risk showerers carry relative to non-showerers |
| PAR / PAF | Cross-checked via Levin's formula vs. the I_total form; quantifies "the theoretical proportion of cases preventable by removing the exposure" |
| 2×2 illustrative example | Practice AR/RR/PAF with illustrative food poisoning teaching numbers (not a real study) |
| Counterfactual | Estimate the expected effect of "removing the exposure" |
| DiD | `daily_cases ~ treated + post + treated:post` (HC3-robust SEs) |
| Parallel trends | Whether the two groups' trends match before the intervention |

**Conclusions**:
- A DAG helps us clarify which variables to control for and which not to
- AR/PAR/PAF quantify an exposure's contribution, but only if the causal relationship holds
- DiD is a quasi-experimental method for evaluating intervention effects, but it requires the parallel trends assumption
- With observational data, causal inference always calls for caution

In the next chapter (Ch13), we make sure all analyses are reproducible → reproducible research.